# Interactive Rollout Demo: Autopoietic Recovery

Generates two publication-quality outputs from the epoch-30 Laplacian checkpoint using CPU or GPU inference only (no training required):

1. **autopoietic_recovery_animated.gif**: Side-by-side animation showing ground truth, model prediction (Otsu-binarised), and difference heatmap, with per-step SSIM overlay. Up to 5 sequences stacked vertically.
2. **fig_belieffield_evolution.pdf/.png**: Grid of the top-8 most-active BeliefField channels across 10 rollout timesteps. Shows how the PDE evolves the internal latent state.

Both outputs go to `paper/figures/`.

## Cell 1: Imports and Setup

Standard scientific stack plus FluidWorld. Publication-quality matplotlib defaults set here. `FIGURES_DIR` controls all output paths.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import Image, display

# -- Paths ------------------------------------------------------------------
PROJECT = Path.cwd().parent
CHECKPOINT = PROJECT / "checkpoints" / "moving_mnist" / "model_epoch_30.pt"
DATA_PATH  = PROJECT / "data" / "mnist_test_seq.npy"
FIGURES_DIR = PROJECT / "paper" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# -- Config -----------------------------------------------------------------
N_SEQUENCES = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# -- Publication-quality matplotlib styling ----------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})


def compute_ssim(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """SSIM between pred and target, both (B, C, H, W) in [0, 1]."""
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    mu_p = F.avg_pool2d(pred, 3, 1, 1)
    mu_t = F.avg_pool2d(target, 3, 1, 1)
    sigma_p  = F.avg_pool2d(pred ** 2, 3, 1, 1) - mu_p ** 2
    sigma_t  = F.avg_pool2d(target ** 2, 3, 1, 1) - mu_t ** 2
    sigma_pt = F.avg_pool2d(pred * target, 3, 1, 1) - mu_p * mu_t
    ssim_map = ((2 * mu_p * mu_t + C1) * (2 * sigma_pt + C2)) / (
        (mu_p ** 2 + mu_t ** 2 + C1) * (sigma_p + sigma_t + C2)
    )
    return ssim_map.mean(dim=(1, 2, 3))


## Cell 2: Load Model and Data

Instantiates `FluidWorldModelV2` with the full feature set (inhibition, memory pump, Hebbian, DeltaNet, Titans) and loads the epoch-30 checkpoint. Then loads Moving MNIST test sequences: initial frame plus 19 ground-truth future frames.

In [ ]:
from fluidworld.core.world_model_v2 import FluidWorldModelV2

model = FluidWorldModelV2(
    in_channels=1,
    d_model=128,
    stimulus_dim=1,
    n_encoder_layers=3,
    max_steps_encoder=6,
    belief_spatial_hw=16,
    n_belief_evolve=3,
    loss_type="bce",
    use_fatigue=False,
    use_inhibition=True,
    use_memory_pump=True,
    use_hebbian=True,
    use_deltanet=True,
    use_titans=True,
)

ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)
state_dict = ckpt.get("model_state_dict", ckpt)
model.load_state_dict(state_dict, strict=False)
model.to(DEVICE).eval()
print(f"Loaded checkpoint: {CHECKPOINT.name}")

# -- Load Moving MNIST sequences --------------------------------------------
raw = np.load(str(DATA_PATH))  # (T, N, 64, 64) or (N, T, 64, 64)
if raw.shape[0] == 20:  # standard layout: (T=20, N, 64, 64)
    raw = raw.transpose(1, 0, 2, 3)

raw = raw[:N_SEQUENCES].astype(np.float32) / 255.0
data = torch.from_numpy(raw).unsqueeze(2).to(DEVICE)  # (N, T, 1, 64, 64)
x_init = data[:, 0]   # (N, 1, 64, 64)
gt_seq = data[:, 1:]  # (N, 19, 1, 64, 64)
print(f"Loaded {N_SEQUENCES} sequences from {DATA_PATH.name}")


## Cell 3: Generate Animated GIF

Runs a 19-step autoregressive rollout, computes per-step SSIM, then builds a side-by-side animation (ground truth / Otsu-binarised prediction / absolute difference heatmap). Predictions are binarised via Otsu thresholding to answer "does the model think a digit is here?", compensating for Laplacian smoothing blur. SSIM is still computed on raw values.

Output: `paper/figures/autopoietic_recovery_animated.gif`

In [ ]:
def _enhance_prediction(frame):
    """Binarise prediction using adaptive Otsu thresholding.

    MNIST is binary (black=0, white=1). Raw predictions are blurry
    due to Laplacian smoothing and fade during autoregressive rollout.
    Otsu finds the optimal threshold per frame, adapting to the
    intensity distribution. Border artifacts (4px) are cropped.
    """
    h, w = frame.shape
    cropped = np.zeros_like(frame)
    interior = frame[4:h-4, 4:w-4]

    if interior.max() - interior.min() < 1e-7:
        return cropped

    # Histogram-based Otsu (no OpenCV dependency)
    hist, bin_edges = np.histogram(interior.ravel(), bins=256,
                                   range=(interior.min(), interior.max()))
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    total = hist.sum()
    if total == 0:
        return cropped

    best_thresh = bin_centers[0]
    best_var = 0
    w0, sum0 = 0, 0
    sum_total = (hist * bin_centers).sum()

    for i in range(len(hist)):
        w0 += hist[i]
        if w0 == 0:
            continue
        w1 = total - w0
        if w1 == 0:
            break
        sum0 += hist[i] * bin_centers[i]
        mu0 = sum0 / w0
        mu1 = (sum_total - sum0) / w1
        between_var = w0 * w1 * (mu0 - mu1) ** 2
        if between_var > best_var:
            best_var = between_var
            best_thresh = bin_centers[i]

    binary = (interior > best_thresh).astype(np.float32)
    cropped[4:h-4, 4:w-4] = binary
    return cropped


# -- Rollout ----------------------------------------------------------------
n_seq = x_init.shape[0]
n_steps = gt_seq.shape[1]  # 19

print("Running rollout...")
stim = torch.zeros(n_seq, 1, device=DEVICE)
with torch.no_grad():
    pred_seq = model.rollout(x_init, stim, n_steps=n_steps)  # (N, 19, 1, H, W)

# Per-step SSIM
ssim_all = torch.zeros(n_seq, n_steps)
for t in range(n_steps):
    ssim_all[:, t] = compute_ssim(
        pred_seq[:, t].to(DEVICE), gt_seq[:, t].to(DEVICE)
    ).cpu()

# Select sequences: best overall first, then fill for diversity
mean_ssim_per_seq = ssim_all.mean(dim=1)
n_show = min(n_seq, 5)
if n_seq <= 5:
    show_idx = list(range(n_seq))
else:
    sorted_idx = mean_ssim_per_seq.argsort(descending=True)
    show_idx = sorted_idx[:n_show].tolist()
show_idx.sort()

print(f"Building animation with {n_show} sequences, {n_steps} steps")

# -- Build animation --------------------------------------------------------
fig, axes = plt.subplots(
    n_show, 3, figsize=(7, 2.0 * n_show + 0.6),
    gridspec_kw={"wspace": 0.05, "hspace": 0.35},
)
if n_show == 1:
    axes = axes[np.newaxis, :]

for j, title in enumerate(["Ground Truth", "Prediction (enhanced)", "Abs. Difference"]):
    axes[0, j].set_title(title, fontsize=9, fontweight="bold")

ims_gt, ims_pred, ims_diff = [], [], []
texts_ssim = []

for row, si in enumerate(show_idx):
    gt0 = gt_seq[si, 0, 0].cpu().numpy()
    pr0 = pred_seq[si, 0, 0].cpu().numpy()
    diff0 = np.abs(gt0 - pr0)

    im_g = axes[row, 0].imshow(gt0, cmap="gray", vmin=0, vmax=1)
    pr0_enhanced = _enhance_prediction(pr0)
    im_p = axes[row, 1].imshow(pr0_enhanced, cmap="gray", vmin=0, vmax=1)
    im_d = axes[row, 2].imshow(diff0, cmap="hot", vmin=0, vmax=0.5)

    ims_gt.append(im_g)
    ims_pred.append(im_p)
    ims_diff.append(im_d)

    for ax in axes[row, :]:
        ax.set_xticks([])
        ax.set_yticks([])

    axes[row, 0].set_ylabel(f"Seq {si}", fontsize=8)

    s = ssim_all[si, 0].item()
    txt = axes[row, 1].text(
        1, 60, f"SSIM: {s:.3f}", fontsize=7,
        color="lime" if s > 0.7 else "yellow" if s > 0.4 else "red",
        fontweight="bold", ha="left", va="bottom",
    )
    texts_ssim.append(txt)

step_text = fig.suptitle("Step 1 / 19", fontsize=11, fontweight="bold", y=0.99)

divider = make_axes_locatable(axes[-1, 2])
cax = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(ims_diff[-1], cax=cax, label="Abs Error")


def update(t):
    for row, si in enumerate(show_idx):
        gt_frame = gt_seq[si, t, 0].cpu().numpy()
        pr_frame = pred_seq[si, t, 0].cpu().numpy()
        diff_frame = np.abs(gt_frame - pr_frame)
        pr_enhanced = _enhance_prediction(pr_frame)
        ims_gt[row].set_data(gt_frame)
        ims_pred[row].set_data(pr_enhanced)
        ims_diff[row].set_data(diff_frame)

        s = ssim_all[si, t].item()
        texts_ssim[row].set_text(f"SSIM: {s:.3f}")
        texts_ssim[row].set_color(
            "lime" if s > 0.7 else "yellow" if s > 0.4 else "red"
        )
    step_text.set_text(f"Step {t + 1} / {n_steps}")
    return ims_gt + ims_pred + ims_diff + texts_ssim + [step_text]


anim = animation.FuncAnimation(
    fig, update, frames=n_steps, interval=400, blit=False,
)

gif_path = FIGURES_DIR / "autopoietic_recovery_animated.gif"
anim.save(str(gif_path), writer="pillow", fps=3)
plt.close(fig)
print(f"Saved -> {gif_path}")

# Display inline
display(Image(filename=str(gif_path)))


## Cell 4: BeliefField State Evolution Figure

Runs a step-by-step rollout on the first sequence, capturing the 16x16 BeliefField spatial state at each step. Selects the top-8 most-active channels (by variance across time and space) and plots them across 10 evenly spaced timesteps. Global colour normalisation for consistent comparison.

The resulting grid reveals PDE diffusion patterns as activations spread and evolve.

Output: `paper/figures/fig_belieffield_evolution.{pdf,png}`

In [ ]:
# Use the first sequence only
x0 = x_init[0:1]  # (1, 1, 64, 64)

print("Running step-by-step rollout for BeliefField capture...")
model.eval()
with torch.no_grad():
    enc_out = model.encode(x0)
    z = enc_out["features"]  # (1, 128, Hf, Wf)

    state = model.belief_field.init_state(1, DEVICE, x0.dtype)
    state = model.belief_field.write(state, z)
    states_over_time = [state.clone()]

    stim = torch.zeros(1, 1, device=DEVICE)
    for t in range(19):
        state = model.belief_field.evolve(state, stimulus=stim)
        z_pred = model.belief_field.read_spatial(
            state, (z.shape[2], z.shape[3])
        )
        x_pred = model.decode_to_pixels(z_pred)
        enc_next = model.encode(x_pred)
        z_next = enc_next["features"]
        state = model.belief_field.write(state, z_next)
        states_over_time.append(state.clone())

# Stack: (20, 128, 16, 16)
all_states = torch.stack(states_over_time, dim=0).squeeze(1).cpu()

# Pick 10 evenly spaced timesteps
timestep_indices = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
timestep_labels = [str(i + 1) for i in timestep_indices]

# Top-8 most active channels (by variance across time and space)
channel_var = all_states.var(dim=(0, 2, 3))  # (128,)
top_channels = channel_var.argsort(descending=True)[:8].numpy()

n_rows = 8
n_cols = len(timestep_indices)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 1.1 + 0.6, n_rows * 1.1 + 0.8),
    gridspec_kw={"wspace": 0.08, "hspace": 0.25},
)

fig.suptitle(
    "BeliefField State Evolution During Rollout",
    fontsize=12, fontweight="bold", y=0.98,
)

# Global normalisation for consistent colourmap
selected_data = all_states[timestep_indices][:, top_channels]
vmin = selected_data.min().item()
vmax = selected_data.max().item()
norm = Normalize(vmin=vmin, vmax=vmax)

for row_i, ch in enumerate(top_channels):
    for col_j, ti in enumerate(timestep_indices):
        ax = axes[row_i, col_j]
        hmap = all_states[ti, ch].numpy()
        ax.imshow(hmap, cmap="viridis", norm=norm, interpolation="nearest")
        ax.set_xticks([])
        ax.set_yticks([])

        if row_i == 0:
            ax.set_title(f"t={timestep_labels[col_j]}", fontsize=7)
        if col_j == 0:
            ax.set_ylabel(f"Ch {ch}", fontsize=7)

# Shared colourbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label="Activation")

fig.text(
    0.5, 0.01,
    "Top-8 most active channels of the 16x16 BeliefField spatial state.\n"
    "PDE diffusion patterns are visible as activations spread and evolve over time.",
    ha="center", fontsize=7, style="italic", wrap=True,
)

for ext in ("pdf", "png"):
    p = FIGURES_DIR / f"fig_belieffield_evolution.{ext}"
    fig.savefig(str(p))
    print(f"Saved -> {p}")

plt.show()


## Done

All outputs saved to `paper/figures/`.